<a href="https://colab.research.google.com/github/jameshphan-png/Group-Exercise-Agentic-AI-in-Customer-Service-Sales-/blob/dev/Customer_Service_Group_Exercise_Part_2_Refund_Policy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**IMPORTANT INFORMATION, READ BELOW**

In [ ]:
#THE FOLLOWING CODE IS A CHATBOT CENTRALIZED AROUND REFUND POLICIES, WHICH IS AN AGENTIC CUSTOMER SERVICE BOT THAT RESPONDS TO INQUIRIES DEDICATED TO THIS SEGMENT.

# ============================================================
#  Acme Corp — Agentic Customer Service Chatbot
#  Uses GeminiAPI - Based on 2.5 Flash Lite
# ============================================================

**Set-Up Company Details & Role Prompt (Refund Policy)**

*   API Call
*   Chat Conversation using LangGraph as memory > Logging conversation in a json file
*   Architecture focuses on the calling of "REFUND_POLICY" function in order to handle various scenarios based on the context of the product. This ensures a professional standard held before ensuring a proper response

**Due the nature of the project's API requests being rate-limited. It is recommended that you refer to the other project file to "run the code"**

Alternative Project File: https://colab.research.google.com/drive/1X4yUyMXIDnqAEQqudRU92dmMT-l9IcI9?usp=sharing

In [1]:
# ── Install & Imports ─────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install", "google-genai", "langgraph", "langchain-core", "-q"], check=True)

import os, json, re, time
from datetime import datetime
from typing import Annotated
from typing_extensions import TypedDict

from google import genai
from google.colab import userdata

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

# ── Set-up Gemini API Client ──────────────────────────────────────────────────
api_key = userdata.get('part2')
client  = genai.Client(api_key=api_key)
MODEL   = "gemini-2.5-flash-lite"

# ── Company Config ────────────────────────────────────────────────────────────
COMPANY_CONFIG = {
    "name"          : "Acme Corp",
    "industry"      : "E-Commerce",
    "support_email" : "refunds@acmecorp.com",
    "support_hours" : "Monday-Friday, 9 AM - 6 PM EST",
    "website"       : "https://www.acmecorp.com/refunds",
}

# ── Refund Policy Rules ───────────────────────────────────────────────────────
REFUND_POLICY = {
    "standard_window_days"  : 30,
    "damaged_window_days"   : 60,
    "restocking_fee_pct"    : 15,
    "free_return_reasons"   : ["defective", "damaged", "wrong item", "not as described"],
    "non_refundable_items"  : ["downloadable software", "gift cards", "personalized items"],
    "refund_methods"        : {
        "original_payment"  : "5-7 business days",
        "store_credit"      : "instant",
    },
    "exchange_available"    : True,
    "return_shipping"       : "Free for defective/wrong items. Customer pays for change-of-mind returns.",
}

# ── Mock Order Database ───────────────────────────────────────────────────────
ORDERS_DB = {
    "ORD-1001": {
        "item"            : "AlphaBook Pro Laptop",
        "status"          : "Delivered",
        "delivered_date"  : "2026-03-01",
        "price"           : 1499,
        "opened"          : True,
        "reason_reported" : None,
    },
    "ORD-1002": {
        "item"            : "GammaAir X Laptop",
        "status"          : "Delivered",
        "delivered_date"  : "2026-02-10",
        "price"           : 1399,
        "opened"          : False,
        "reason_reported" : None,
    },
    "ORD-1003": {
        "item"            : "NanoEdge Flex Laptop",
        "status"          : "Delivered",
        "delivered_date"  : "2026-01-05",
        "price"           : 1699,
        "opened"          : True,
        "reason_reported" : "defective",
    },
    "ORD-1004": {
        "item"            : "Downloadable Software Suite",
        "status"          : "Delivered",
        "delivered_date"  : "2026-03-10",
        "price"           : 199,
        "opened"          : True,
        "reason_reported" : None,
    },
}

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = f"""
You are Riley, a professional and empathetic refund policy specialist for {COMPANY_CONFIG['name']},
an {COMPANY_CONFIG['industry']} company.

YOUR RESPONSIBILITIES:
1. Help customers understand the refund and return policy clearly and honestly.
2. Assess refund eligibility based on the order data and policy rules provided to you.
3. Guide customers through the refund or exchange process step by step.
4. Handle edge cases (damaged items, non-refundable products, late requests) with empathy.

REFUND POLICY RULES YOU MUST FOLLOW:
- Standard return window   : {REFUND_POLICY['standard_window_days']} days from delivery date.
- Damaged/defective window : {REFUND_POLICY['damaged_window_days']} days from delivery date.
- Restocking fee           : {REFUND_POLICY['restocking_fee_pct']}% for opened, non-defective items.
- Free returns             : {', '.join(REFUND_POLICY['free_return_reasons'])}.
- Non-refundable items     : {', '.join(REFUND_POLICY['non_refundable_items'])}.
- Refund to original payment takes {REFUND_POLICY['refund_methods']['original_payment']}.
- Store credit is issued instantly as an alternative.
- Exchanges are available as an alternative to refunds.
- Return shipping          : {REFUND_POLICY['return_shipping']}

BEHAVIOUR RULES:
- Always address the customer by name once you learn it.
- Be empathetic — refund situations are often stressful for customers.
- Never promise outcomes you cannot guarantee (e.g. "your refund will definitely arrive tomorrow").
- If a case needs a human review (e.g. disputed eligibility), say a specialist will email within 1 business day.
- Remember all details the customer shares across the entire conversation.
- Keep responses to 3-5 sentences unless the customer needs a detailed walkthrough.

COMPANY DETAILS:
- Refunds email : {COMPANY_CONFIG['support_email']}
- Support hours : {COMPANY_CONFIG['support_hours']}
- Refund portal : {COMPANY_CONFIG['website']}
""".strip()

# ── Intent Detection ──────────────────────────────────────────────────────────
INTENT_KEYWORDS = {
    "check_eligibility" : ["eligible", "qualify", "can i return", "can i get a refund",
                           "do i get", "am i able", "ord-"],
    "policy_question"   : ["policy", "how long", "window", "days", "restocking", "fee",
                           "free return", "shipping", "non-refundable", "what items"],
    "start_refund"      : ["start", "begin", "initiate", "process", "submit", "how do i",
                           "steps", "request a refund", "request a return"],
    "exchange"          : ["exchange", "swap", "replace", "replacement", "different item"],
    "refund_status"     : ["status", "where is my refund", "when will i", "still waiting",
                           "not received", "how long does it take"],
    "damaged"           : ["damaged", "defective", "broken", "not working", "wrong item",
                           "not as described", "faulty"],
}

def detect_intent(message: str) -> str:
    msg = message.lower()
    for intent, keywords in INTENT_KEYWORDS.items():
        if any(kw in msg for kw in keywords):
            return intent
    return "general"

# ── Order Lookup & Eligibility Check ─────────────────────────────────────────
def lookup_order_with_eligibility(message: str) -> str | None:
    match = re.search(r'ORD-\d+', message, re.IGNORECASE)
    if not match:
        return None

    order_id = match.group().upper()
    if order_id not in ORDERS_DB:
        return f"[SYSTEM NOTE: Order {order_id} not found in the system. Ask the customer to double-check.]"

    o          = ORDERS_DB[order_id]
    today      = datetime.today().date()
    delivered  = datetime.strptime(o["delivered_date"], "%Y-%m-%d").date()
    days_since = (today - delivered).days

    for nr_item in REFUND_POLICY["non_refundable_items"]:
        if nr_item.lower() in o["item"].lower():
            return _format_order_note(
                order_id, o, days_since,
                eligibility = "NOT ELIGIBLE — this item is non-refundable per policy.",
                restocking  = "N/A",
                window_used = "N/A",
                notes       = "Inform the customer politely that this category cannot be refunded.",
            )

    is_damaged = o.get("reason_reported") in REFUND_POLICY["free_return_reasons"]
    window     = REFUND_POLICY["damaged_window_days"] if is_damaged else REFUND_POLICY["standard_window_days"]
    window_label = f"{window}-day ({'damaged/defective' if is_damaged else 'standard'})"

    if days_since > window:
        return _format_order_note(
            order_id, o, days_since,
            eligibility = f"NOT ELIGIBLE — {days_since} days since delivery exceeds the {window}-day window.",
            restocking  = "N/A",
            window_used = window_label,
            notes       = "Sympathise with the customer and offer to escalate for a manual review if they push back.",
        )

    if is_damaged or not o["opened"]:
        fee_note = "No restocking fee (defective/unopened)."
    else:
        fee      = round(o["price"] * REFUND_POLICY["restocking_fee_pct"] / 100, 2)
        fee_note = f"{REFUND_POLICY['restocking_fee_pct']}% restocking fee applies = ${fee} deducted from refund."

    return _format_order_note(
        order_id, o, days_since,
        eligibility = f"ELIGIBLE — {days_since} days since delivery, within the {window}-day window.",
        restocking  = fee_note,
        window_used = window_label,
        notes       = "Walk the customer through next steps: submit via portal or email.",
    )


def _format_order_note(order_id, o, days_since, eligibility, restocking, window_used, notes) -> str:
    return (
        f"[SYSTEM NOTE - Refund eligibility for {order_id}:\n"
        f"  Item           : {o['item']}\n"
        f"  Price          : ${o['price']}\n"
        f"  Delivered      : {o['delivered_date']} ({days_since} days ago)\n"
        f"  Opened         : {'Yes' if o['opened'] else 'No'}\n"
        f"  Reported issue : {o.get('reason_reported') or 'None'}\n"
        f"  Window applied : {window_used}\n"
        f"  Eligibility    : {eligibility}\n"
        f"  Restocking fee : {restocking}\n"
        f"  Agent notes    : {notes}\n"
        f"Use these exact details in your reply. Do NOT reveal internal agent notes to the customer.]"
    )


def build_policy_context() -> str:
    return (
        "[SYSTEM NOTE - Full refund policy summary:\n"
        f"  Standard return window   : {REFUND_POLICY['standard_window_days']} days from delivery\n"
        f"  Damaged/defective window : {REFUND_POLICY['damaged_window_days']} days from delivery\n"
        f"  Restocking fee           : {REFUND_POLICY['restocking_fee_pct']}% for opened, non-defective items\n"
        f"  Free return reasons      : {', '.join(REFUND_POLICY['free_return_reasons'])}\n"
        f"  Non-refundable items     : {', '.join(REFUND_POLICY['non_refundable_items'])}\n"
        f"  Refund to original card  : {REFUND_POLICY['refund_methods']['original_payment']}\n"
        f"  Store credit             : {REFUND_POLICY['refund_methods']['store_credit']}\n"
        f"  Return shipping          : {REFUND_POLICY['return_shipping']}\n"
        f"  Exchanges available      : {'Yes' if REFUND_POLICY['exchange_available'] else 'No'}\n"
        "Use this to answer any policy question accurately.]"
    )


def build_process_context() -> str:
    return (
        "[SYSTEM NOTE - Refund/return process steps to share with customer:\n"
        f"  Step 1 : Visit {COMPANY_CONFIG['website']} and log in.\n"
        "  Step 2 : Click 'Start a Return' and enter your order number.\n"
        "  Step 3 : Select your return reason from the dropdown.\n"
        "  Step 4 : Print the prepaid label (if eligible for free return) or ship at your cost.\n"
        "  Step 5 : Drop off the package at any carrier location.\n"
        "  Step 6 : Refund will be processed within 3-5 business days of us receiving the item.\n"
        f"  Alternatively email {COMPANY_CONFIG['support_email']} with order number and reason.\n"
        "Walk the customer through these steps clearly.]"
    )

# ── Conversation Logger ───────────────────────────────────────────────────────
class ConversationLogger:
    def __init__(self, session_id: str):
        self.session_id = session_id
        self.data = {
            "session_id" : session_id,
            "started_at" : datetime.now().isoformat(),
            "company"    : COMPANY_CONFIG["name"],
            "turns"      : 0,
            "messages"   : [],
        }

    def log(self, role: str, content: str, intent: str = ""):
        entry = {"timestamp": datetime.now().isoformat(), "role": role, "content": content}
        if intent:
            entry["intent"] = intent
        self.data["messages"].append(entry)
        if role == "user":
            self.data["turns"] += 1

    def save(self):
        self.data["ended_at"] = datetime.now().isoformat()
        log_file = "support_log.json"
        try:
            existing = json.load(open(log_file)) if os.path.exists(log_file) else []
            existing.append(self.data)
            json.dump(existing, open(log_file, "w"), indent=2)
            print(f"\n  Session saved to {log_file}  (ID: {self.session_id}, turns: {self.data['turns']})")
        except Exception as e:
            print(f"\n  Could not save log: {e}")



# ── LangGraph Setup ──────────────────────────────────────────────────────────


class AgentState(TypedDict):
    """
    Graph state persisted by MemorySaver after every turn.

    messages — full conversation history; the add_messages reducer
               *appends* incoming messages rather than overwriting,
               so no prior turn is ever lost.
    """
    messages: Annotated[list, add_messages]


def gemini_node(state: AgentState) -> dict:
    """
    Single graph node: converts LangGraph message objects into the flat
    prompt string Gemini expects, calls the API, and returns the new
    AIMessage. MemorySaver snapshots the updated state automatically
    after this node returns.
    """
    full_prompt = SYSTEM_PROMPT + "\n\n"
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage):
            full_prompt += f"Customer: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            full_prompt += f"Riley: {msg.content}\n"
    full_prompt += "Riley:"

    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=full_prompt,
            )
            reply = response.text.strip()
            # Return only the *new* message — add_messages merges it into state
            return {"messages": [AIMessage(content=reply)]}
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                wait = 2 ** attempt
                print(f"  Rate limited, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise


def build_graph() -> StateGraph:
    """
    Compile a minimal START → riley → END graph with MemorySaver attached.

    MemorySaver checkpoints AgentState (the full message list) after every
    .invoke() call, keyed by thread_id. Passing the same thread_id on the
    next turn restores the entire conversation automatically — no manual
    history list needed.

    To persist across process restarts later, swap MemorySaver for
    SqliteSaver or PostgresSaver with zero other code changes.
    """
    memory  = MemorySaver()
    builder = StateGraph(AgentState)
    builder.add_node("riley", gemini_node)
    builder.add_edge(START, "riley")
    builder.add_edge("riley", END)
    return builder.compile(checkpointer=memory)


# One shared graph instance — MemorySaver lives inside it
GRAPH = build_graph()


def invoke_graph(thread_id: str, human_content: str) -> str:
    """
    Send one user turn to the graph and return Riley's reply.

    thread_id     — MemorySaver key; reusing the same ID restores history
    human_content — the (possibly context-augmented) user message
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = GRAPH.invoke(
        {"messages": [HumanMessage(content=human_content)]},
        config=config,
    )
    return result["messages"][-1].content


# ── Main Chat Session ─────────────────────────────────────────────────────────
def run_chat_session():
    # thread_id is the MemorySaver checkpoint key — unique per session
    thread_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    logger    = ConversationLogger(session_id=thread_id)

    print("=" * 60)
    print(f"  {COMPANY_CONFIG['name']}  -  Refund & Returns Support")
    print(f"  {COMPANY_CONFIG['support_hours']}")
    print(f"  {COMPANY_CONFIG['support_email']}")
    print(f"  Session ID (MemorySaver thread): {thread_id}")
    print("=" * 60)
    print("  Ask about refunds, returns, or your order status.")
    print("  Type 'done' at any time to end the session.")
    print("-" * 60)

    # ── Greeting turn ─────────────────────────────────────────────────────────
    greeting = invoke_graph(
        thread_id,
        "Greet the customer warmly, introduce yourself as a refund specialist, "
        "and ask how you can help.",
    )
    logger.log("assistant", greeting)
    print(f"\n  Riley: {greeting}\n")

    # ── Conversation loop ──────────────────────────────────────────────────────
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("done", "quit", "exit", "bye", "goodbye"):
            break

        intent = detect_intent(user_input)
        logger.log("user", user_input, intent=intent)

        # ── Augment message with refund / policy / order context ───────────
        if intent == "check_eligibility":
            note      = lookup_order_with_eligibility(user_input)
            augmented = (
                f"{user_input}\n\n{note}" if note else
                f"{user_input}\n\n[SYSTEM NOTE: No order number detected. "
                "Ask the customer for their order number (format: ORD-XXXX) before proceeding.]"
            )

        elif intent == "policy_question":
            augmented = f"{user_input}\n\n{build_policy_context()}"

        elif intent == "start_refund":
            augmented = f"{user_input}\n\n{build_process_context()}"

        elif intent == "exchange":
            augmented = (
                f"{user_input}\n\n[SYSTEM NOTE: Exchanges are available. "
                "Instruct the customer to start a return via the portal and select 'Exchange' as the reason. "
                f"Portal: {COMPANY_CONFIG['website']}. "
                "If they have an order number, ask for it to check eligibility first.]"
            )

        elif intent == "refund_status":
            augmented = (
                f"{user_input}\n\n[SYSTEM NOTE: Refund timelines are "
                f"{REFUND_POLICY['refund_methods']['original_payment']} to original payment, "
                f"or {REFUND_POLICY['refund_methods']['store_credit']} for store credit. "
                "If the customer says they have waited longer than this, offer to escalate to a specialist "
                "who will follow up within 1 business day.]"
            )

        elif intent == "damaged":
            augmented = (
                f"{user_input}\n\n[SYSTEM NOTE: Damaged or defective items qualify for the "
                f"{REFUND_POLICY['damaged_window_days']}-day return window with NO restocking fee and FREE return shipping. "
                "Ask for the order number if not already provided so you can confirm eligibility. "
                "Express empathy first before diving into logistics.]"
            )

        else:
            augmented = user_input

        # ── Send to graph — MemorySaver restores full history via thread_id ─
        reply = invoke_graph(thread_id, augmented)
        logger.log("assistant", reply)

        print(f"\n  Riley: {reply}\n")
        print("-" * 60)

    # ── Closing turn ──────────────────────────────────────────────────────────
    closing = invoke_graph(
        thread_id,
        "The customer is ending the session. Give a warm, 1-2 sentence goodbye "
        "and mention they can email for further help.",
    )
    logger.log("assistant", closing)
    print(f"\n  Riley: {closing}\n")
    print("=" * 60)

    logger.save()

    # ── Show what MemorySaver stored for this thread ───────────────────────
    snapshot  = GRAPH.get_state({"configurable": {"thread_id": thread_id}})
    msg_count = len(snapshot.values["messages"])
    print(f"\n  [MemorySaver] {msg_count} messages checkpointed for thread '{thread_id}'")


# ── Entry Point ───────────────────────────────────────────────────────────────
while True:
    run_chat_session()
    again = input("\n  Start a new support session? (yes / no): ").strip().lower()
    if again not in ("yes", "y"):
        print(f"\n  Thank you for contacting {COMPANY_CONFIG['name']}. Have a great day!\n")
        break

  Acme Corp  -  Refund & Returns Support
  Monday-Friday, 9 AM - 6 PM EST
  refunds@acmecorp.com
  Session ID (MemorySaver thread): 20260323_004424
  Ask about refunds, returns, or your order status.
  Type 'done' at any time to end the session.
------------------------------------------------------------

  Riley: Hello there! My name is Riley, and I'm a refund specialist here at Acme Corp. I'm here to help you navigate our refund and return policies and assist you with any questions or issues you might have. How can I help you today?

You: Hello! I would like to know the refund policy

  Riley: I can certainly help you understand our refund policy! Generally, you have 30 days from the delivery date to return most items for a refund. If an item is defective or damaged, that window extends to 60 days. There's a 15% restocking fee for opened, non-defective items. For defective, damaged, or incorrect items, returns are free, and we also offer exchanges.

---------------------------------